# WAVES: The Radio and Plasma Wave Investigation on the WIND Spacecraft — Implementation / WIND/WAVES 구현

**Paper**: Bougeret, J.-L., Kaiser, M. L., Kellogg, P. J., Manning, R., Goetz, K., Monson, S. J., Monge, N., Friel, L., Meetre, C. A., Perche, C., Sitruk, L., and Hoang, S. (1995). *WAVES: The Radio and Plasma Wave Investigation on the WIND Spacecraft*. Space Science Reviews, **71**, 231–263. DOI: 10.1007/BF00751331

**English.** This notebook reproduces four signature algorithms enabled by WIND/WAVES with synthetic data and `numpy`/`scipy`/`matplotlib`:
1. Type II / III burst dynamic-spectrum simulation including frequency drift through a coronal density model.
2. Langmuir-wave detection in the solar wind by tracking the local plasma frequency — mimicking the TNR (Thermal Noise Receiver) onboard neural-network's role.
3. Goniopolarimetric direction-finding from a four-antenna spinning-spacecraft array (RAD1/RAD2 SUM-mode synthesis).
4. IP shock arrival-time prediction at L1 from the type II onset frequency drift.

**Korean.** 이 노트북은 WIND/WAVES 의 대표 알고리즘 네 개를 합성 데이터와 `numpy`/`scipy`/`matplotlib`로 재현한다:
1. 코로나 밀도 모델을 통한 주파수 드리프트를 포함한 II형/III형 전파폭발 동적 스펙트럼 시뮬레이션.
2. 국소 플라즈마 주파수 추적을 통한 태양풍 내 Langmuir 파 검출 — TNR 온보드 신경망 역할 모사.
3. 4안테나 회전 위성 배열로부터 goniopolarimetric 방향 탐지(RAD1/RAD2 SUM 모드 합성).
4. II형 폭발 시작 주파수 드리프트로부터 L1 행성간 충격파 도착 시각 예측.

Conda env: `study-with-ai`.

In [ ]:
"""Imports and global plot configuration."""

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.optimize import least_squares

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
rng = np.random.default_rng(seed=19951101)  # WIND launch date

# Physical constants (SI / cgs hybrid commonly used in solar radio physics).
C_LIGHT_KMS = 299792.458         # speed of light, km / s
R_SUN_KM = 6.957e5               # solar radius, km
AU_KM = 1.495978707e8            # astronomical unit, km
AU_RSUN = AU_KM / R_SUN_KM       # 1 AU expressed in R_sun (~215)

FP_COEFF_KHZ = 8.980             # f_p [kHz] = 8.980 * sqrt(n_e [cm^-3])


def plasma_frequency_khz(n_e_cm3: np.ndarray) -> np.ndarray:
    """Return the local electron plasma frequency in kHz.

    Args:
        n_e_cm3: Electron number density in cm^-3.

    Returns:
        Plasma frequency in kHz, using f_p[Hz] = (1/2 pi) * sqrt(n e^2 / eps0 m_e).
    """
    return FP_COEFF_KHZ * np.sqrt(np.asarray(n_e_cm3, dtype=float))


print(f"f_p at n_e=5 cm^-3: {plasma_frequency_khz(5.0):.2f} kHz (typical 1 AU)")
print(f"f_p at n_e=650 cm^-3: {plasma_frequency_khz(650.0):.1f} kHz (10 R_sun corona)")

## Part 1: Type II / III Burst Dynamic Spectrum Simulation / II형·III형 전파폭발 동적 스펙트럼 시뮬레이션

**English.** A type III burst is generated by a relativistic (~0.3c) electron beam exciting Langmuir waves at the local plasma frequency `f_p(r)` along its trajectory; mode-conversion produces electromagnetic radiation at `f_p` (fundamental) and `2 f_p` (harmonic). As the beam moves outward through decreasing density, the emission frequency *drifts down*. We adopt a Saito–Newkirk-style coronal density profile (notes §4.7),

$$ n_e(r) = A\, r^{-2} + B\, r^{-4}, \qquad A = 6.5\times10^{4}\,\text{cm}^{-3}\,R_\odot^2,\; B = 4\times10^{4}\,\text{cm}^{-3}\,R_\odot^4. $$

and follow a beam at `v_b = 0.3 c` from `r_0 = 1.5 R_sun`. Type II bursts come from CME-driven shocks moving more slowly (here 1500 km/s) and exhibit a much slower drift.

**Korean.** III형 전파폭발은 상대론적(약 0.3c) 전자 빔이 경로를 따라 국소 플라즈마 주파수 `f_p(r)`에서 Langmuir 파를 여기하고, 모드 변환으로 `f_p`(기본)와 `2 f_p`(고조파) EM 복사가 발생한다. 빔이이 밀도가 낮은 바깥으로 이동함에 따라 방출 주파수가 하향 드리프트한다. Saito–Newkirk형 코로나 밀도를 사용하며 II형은 CME 주도 충격파(여기서는 1500 km/s)에 의해 더 느린 드리프트를 보인다.

In [ ]:
"""Coronal/heliospheric density model and beam-driven dynamic spectrum generation."""


def coronal_density(r_rsun: np.ndarray) -> np.ndarray:
    """Saito-Newkirk style two-term electron density model.

    Args:
        r_rsun: Heliocentric distance in solar radii.

    Returns:
        Electron number density in cm^-3, smoothly transitioning to a
        Leblanc et al. style 1/r^2 tail beyond a few R_sun toward 1 AU.
    """
    r = np.asarray(r_rsun, dtype=float)
    coronal = 6.5e4 / r**2 + 4.0e4 / r**4
    # Add a constant n_e ~ 5 cm^-3 floor for the L1 solar wind plateau.
    floor = 5.0 * (AU_RSUN / np.maximum(r, 1.0))**2
    return np.maximum(coronal, floor)


def beam_trajectory_radius(t_seconds: np.ndarray, v_beam_c: float, r0_rsun: float) -> np.ndarray:
    """Compute heliocentric radius of an electron beam at time t.

    Args:
        t_seconds: Times (s) since beam release.
        v_beam_c: Beam speed as a fraction of light speed.
        r0_rsun: Initial radius in R_sun.

    Returns:
        Heliocentric distance in R_sun, clipped to be >= r0.
    """
    v_kms = v_beam_c * C_LIGHT_KMS
    return r0_rsun + v_kms * np.asarray(t_seconds) / R_SUN_KM


def shock_trajectory_radius(t_seconds: np.ndarray, v_shock_kms: float, r0_rsun: float) -> np.ndarray:
    """Compute heliocentric radius of a CME-driven shock."""
    return r0_rsun + v_shock_kms * np.asarray(t_seconds) / R_SUN_KM


def dynamic_spectrum(
    t_grid_s: np.ndarray,
    f_grid_khz: np.ndarray,
    drive_freqs_khz: np.ndarray,
    intensity: float = 1.0,
    line_width_rel: float = 0.04,
    harmonic: bool = True,
) -> np.ndarray:
    """Build a synthetic dynamic spectrum (time x frequency).

    Each time step has emission concentrated at `drive_freqs_khz[t]` (fundamental)
    and twice that (harmonic), modeled as Gaussian lines on a log-frequency axis.

    Args:
        t_grid_s: Time axis in seconds.
        f_grid_khz: Frequency axis in kHz.
        drive_freqs_khz: Per-time fundamental emission frequency (kHz).
        intensity: Peak intensity (arbitrary units).
        line_width_rel: Relative line width sigma/f.
        harmonic: If True, also add the 2 f_p harmonic at half intensity.

    Returns:
        2D array shaped (len(f_grid_khz), len(t_grid_s)).
    """
    log_f = np.log(f_grid_khz)[:, None]
    log_drive = np.log(drive_freqs_khz)[None, :]
    spec = intensity * np.exp(-0.5 * ((log_f - log_drive) / line_width_rel)**2)
    if harmonic:
        log_h = np.log(2.0 * drive_freqs_khz)[None, :]
        spec = spec + 0.5 * intensity * np.exp(-0.5 * ((log_f - log_h) / line_width_rel)**2)
    return spec


# Type III burst: 0.3c beam from 1.5 R_sun.
t_iii = np.linspace(0.1, 600.0, 600)  # 10 minutes of evolution
r_iii = beam_trajectory_radius(t_iii, v_beam_c=0.3, r0_rsun=1.5)
n_iii = coronal_density(r_iii)
fp_iii = plasma_frequency_khz(n_iii)

# Type II burst: 1500 km/s shock from 2 R_sun, starts 5 min later.
t_ii_local = np.linspace(0.1, 1800.0, 600)  # 30 min after onset
r_ii = shock_trajectory_radius(t_ii_local, v_shock_kms=1500.0, r0_rsun=2.0)
n_ii = coronal_density(r_ii)
fp_ii = plasma_frequency_khz(n_ii)

print(f"Type III: f_p drifts from {fp_iii[0]:.0f} kHz to {fp_iii[-1]:.0f} kHz over {t_iii[-1]:.0f} s")
print(f"Type II:  f_p drifts from {fp_ii[0]:.0f} kHz to {fp_ii[-1]:.0f} kHz over {t_ii_local[-1]:.0f} s")

In [ ]:
"""Render a combined dynamic spectrum spanning RAD2 + RAD1 + TNR bands."""

# Master time axis (0 .. 30 min) and log-spaced WAVES frequency axis.
t_master = np.linspace(0.0, 1800.0, 901)
f_axis_khz = np.logspace(np.log10(4.0), np.log10(13825.0), 256)  # TNR_low..RAD2_high

fp_iii_master = np.interp(t_master, t_iii, fp_iii, left=np.nan, right=fp_iii[-1])
ii_onset_s = 300.0  # type II onset
fp_ii_master = np.full_like(t_master, np.nan)
valid_ii = t_master >= ii_onset_s
fp_ii_master[valid_ii] = np.interp(t_master[valid_ii] - ii_onset_s, t_ii_local, fp_ii)

# Replace NaN with a very small frequency so the Gaussian sits off the visible band.
fp_iii_drive = np.where(np.isfinite(fp_iii_master), fp_iii_master, 1e-3)
fp_ii_drive = np.where(np.isfinite(fp_ii_master), fp_ii_master, 1e-3)

spec_iii = dynamic_spectrum(t_master, f_axis_khz, fp_iii_drive, intensity=1.0)
spec_ii = dynamic_spectrum(t_master, f_axis_khz, fp_ii_drive, intensity=0.6)
noise = 0.02 * rng.standard_normal(spec_iii.shape)
spec_total = np.clip(spec_iii + spec_ii + noise, 0.0, None)
spec_db = 10.0 * np.log10(spec_total + 1e-3)

fig, ax = plt.subplots(figsize=(12, 7))
im = ax.pcolormesh(
    t_master / 60.0, f_axis_khz, spec_db,
    shading='auto', cmap='inferno', vmin=-25, vmax=5,
)
ax.set_yscale('log')
ax.set_xlabel('Time since flare onset (min)')
ax.set_ylabel('Frequency (kHz)')
ax.set_title('Synthetic WAVES dynamic spectrum: type III (fast drift) + type II (slow drift)')
for f_band, label in [(4.0, 'TNR low'), (256.0, 'TNR/RAD1 boundary'),
                       (1040.0, 'RAD1/RAD2 boundary'), (13825.0, 'RAD2 high')]:
    ax.axhline(f_band, color='cyan', lw=0.5, ls='--', alpha=0.6)
    ax.text(0.02, f_band * 1.05, label, transform=ax.get_yaxis_transform(),
            color='cyan', fontsize=8)
fig.colorbar(im, ax=ax, label='Intensity (dB, arb.)')
plt.tight_layout()
plt.show()

## Part 2: Langmuir-wave Detection by Plasma-Frequency Tracking / 플라즈마 주파수 추적을 통한 Langmuir 파 검출

**English.** Type III electron beams drive intense Langmuir waves *in situ* at the local plasma frequency. The TNR sees a sharp, narrow line at `f_p` rising above a smooth thermal-noise quasi-thermal continuum. We mock a 30-minute TNR time series with a slowly-varying solar-wind density (so `f_p` drifts ~18–24 kHz), inject episodic Langmuir wave packets, and reproduce the WAVES neural-network role: a peak-tracking algorithm that locks onto the plasma line and *selects* the TNR band containing it.

**Korean.** III형 전자 빔이은 국소 플라즈마 주파수에서 강한 Langmuir 파를 일으킨다. TNR은 매끄러운 quasi-thermal 연속 점 위에 손는 `f_p` 선을 본다. 태양풍 밀도가 서서히 변하는 30분 동안의 TNR 시계열을 생성하고(`f_p` 약 18–24 kHz), 국부적으로 Langmuir 파 패킷을 주입한 뒤 WAVES 신경망이 담당했던 역할 — 플라즈마 라인을 킨아 해당 TNR 밴드를 선택하는 알고리즘 — 을 재현한다.

In [ ]:
"""Generate synthetic TNR data with a drifting plasma line and Langmuir wave packets."""

TNR_BANDS = {
    'A': (4.0, 16.0),
    'B': (8.0, 32.0),
    'C': (16.0, 64.0),
    'D': (32.0, 128.0),
    'E': (64.0, 256.0),
}


def tnr_log_channels(band: tuple[float, float], n_channels: int = 32) -> np.ndarray:
    """Return logarithmically spaced channel center frequencies for a TNR band."""
    f_lo, f_hi = band
    return np.geomspace(f_lo, f_hi, n_channels)


def quasi_thermal_noise(f_khz: np.ndarray, fp_khz: float, tnr_floor_db: float = -120.0) -> np.ndarray:
    """Synthetic quasi-thermal-noise spectrum following Meyer-Vernet & Perche (1989).

    Below f_p the spectrum drops; near f_p there is a peak (thermal plasma line);
    above f_p it falls as 1/f^3.

    Args:
        f_khz: Frequency grid (kHz).
        fp_khz: Plasma frequency (kHz).
        tnr_floor_db: Receiver noise floor (dB referenced to V^2 / Hz).

    Returns:
        Noise power-density spectrum on the f_khz grid, linear units.
    """
    x = f_khz / fp_khz
    line = 1.0 / ((x - 1.0)**2 + 0.0025)  # narrow Lorentzian peak at f_p
    suppress = np.where(x < 1.0, 0.05 * x**2, 1.0 / x**3)
    spec = 1e-9 * (line + 50.0 * suppress)
    floor = 10**(tnr_floor_db / 10.0)
    return spec + floor


n_time_samples = 600
t_tnr = np.linspace(0.0, 30.0 * 60.0, n_time_samples)  # 30 minutes, ~3 s cadence
n_e_solar_wind = 5.0 + 1.5 * np.sin(2 * np.pi * t_tnr / 1800.0) + 0.3 * rng.standard_normal(n_time_samples)
fp_solar_wind = plasma_frequency_khz(np.maximum(n_e_solar_wind, 0.5))

all_channels = np.concatenate([tnr_log_channels(b) for b in TNR_BANDS.values()])
all_channels = np.unique(all_channels)

spec_tnr = np.zeros((all_channels.size, n_time_samples))
for k, fp in enumerate(fp_solar_wind):
    spec_tnr[:, k] = quasi_thermal_noise(all_channels, fp)

# Inject Langmuir wave packet bursts at random times ~10x stronger than f_p line.
n_bursts = 8
burst_times = rng.choice(np.arange(50, n_time_samples - 50), size=n_bursts, replace=False)
for tb in burst_times:
    fp = fp_solar_wind[tb]
    duration = rng.integers(2, 6)
    line_idx = np.argmin(np.abs(all_channels - fp))
    width = 2  # channels
    sl = slice(max(0, line_idx - width), line_idx + width + 1)
    spec_tnr[sl, tb:tb + duration] *= 30.0

spec_tnr_db = 10.0 * np.log10(spec_tnr / spec_tnr.mean())

In [ ]:
"""Plasma-frequency tracker: simple peak-pick used as a stand-in for the TNR neural network."""


def track_plasma_line(
    spec_db: np.ndarray,
    f_khz: np.ndarray,
    fp_prior: float | None = None,
    smoothing: int = 3,
) -> np.ndarray:
    """Track the plasma line in time using nearest-peak continuation.

    Mimics the WAVES TNR onboard neural network whose role is to identify the
    channel holding the plasma line so it can be sent at full resolution.

    Args:
        spec_db: 2D spectrum (channels x time) in dB.
        f_khz: Channel center frequencies (kHz).
        fp_prior: Optional starting estimate (kHz).
        smoothing: Median window in time for the recovered track.

    Returns:
        Estimated f_p time series (kHz).
    """
    n_t = spec_db.shape[1]
    fp_track = np.zeros(n_t)
    if fp_prior is None:
        fp_prior = f_khz[np.argmax(spec_db[:, 0])]
    fp_prev = fp_prior
    for k in range(n_t):
        # Restrict search to within +/- 30% of previous estimate (NN-style continuity).
        mask = (f_khz >= 0.7 * fp_prev) & (f_khz <= 1.3 * fp_prev)
        if not mask.any():
            mask = np.ones_like(f_khz, dtype=bool)
        local_spec = spec_db[mask, k]
        local_f = f_khz[mask]
        idx = int(np.argmax(local_spec))
        fp_track[k] = local_f[idx]
        fp_prev = 0.6 * fp_prev + 0.4 * fp_track[k]  # leaky integrator
    if smoothing > 1:
        fp_track = signal.medfilt(fp_track, kernel_size=smoothing)
    return fp_track


fp_estimated = track_plasma_line(spec_tnr_db, all_channels, fp_prior=fp_solar_wind[0])
n_e_estimated = (fp_estimated / FP_COEFF_KHZ)**2  # invert plasma-frequency relation

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True,
                          gridspec_kw={'height_ratios': [3, 1]})

im = axes[0].pcolormesh(t_tnr / 60.0, all_channels, spec_tnr_db,
                         shading='auto', cmap='magma', vmin=-30, vmax=30)
axes[0].plot(t_tnr / 60.0, fp_solar_wind, 'c--', lw=1.5, label='True f_p')
axes[0].plot(t_tnr / 60.0, fp_estimated, 'w-', lw=1.0, alpha=0.7, label='Tracked f_p')
axes[0].scatter(t_tnr[burst_times] / 60.0, fp_solar_wind[burst_times], s=80,
                facecolors='none', edgecolors='lime', label='Langmuir bursts')
axes[0].set_yscale('log')
axes[0].set_ylabel('Frequency (kHz)')
axes[0].set_title('Synthetic TNR spectrogram with plasma-line tracking')
axes[0].legend(loc='upper right', fontsize=9)
fig.colorbar(im, ax=axes[0], label='Power (dB rel.)')

axes[1].plot(t_tnr / 60.0, n_e_solar_wind, 'k-', label='True n_e')
axes[1].plot(t_tnr / 60.0, n_e_estimated, 'r--', label='From tracked f_p')
axes[1].set_xlabel('Time (min)')
axes[1].set_ylabel('n_e (cm$^{-3}$)')
axes[1].legend(loc='upper right', fontsize=9)
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

rms_err_khz = np.sqrt(np.mean((fp_estimated - fp_solar_wind)**2))
print(f'RMS plasma-frequency tracking error: {rms_err_khz:.2f} kHz')
print(f'Equivalent density error: {rms_err_khz / FP_COEFF_KHZ * 2 * fp_solar_wind.mean() / FP_COEFF_KHZ:.2f} cm^-3 (1-sigma)')

## Part 3: Four-Antenna Goniopolarimetry / 4안테나 goniopolarimetry

**English.** WAVES uses three orthogonal electric dipoles (Ex, Ey, Ez) and synthesizes inclined-dipole channels (S, S′, Z) via SUM-mode phase shifting. Here we treat the spinning spacecraft as effectively a four-antenna array for direction finding. Given an incoming plane-wave from celestial coordinates `(theta, phi)`, the antenna response on each of four virtual elements rotated through one spin period (3 s) is

$$ V_i(t) = E_0 \cos(\omega t + \psi_i) (\hat{\mathbf e}_i \cdot \hat{\mathbf k}_\perp) + n_i(t), $$

and a least-squares fit recovers `(theta, phi)`. The angular precision improves with `sqrt(N_samples * SNR)`.

**Korean.** WAVES는 세 직교 전기 쌍극자(Ex, Ey, Ez)를 사용하며 SUM 모드 위상 시프트로 경사 쌍극자 채널(S, S′, Z)을 합성한다. 여기서는 회전하는 위성을 방향 탐지용 4안테나 배열로 취급한다. 천구 좌표 `(theta, phi)`의 평면파에 대해 각 안테나의 이던 움직임을 최소제곱으로 적합하여 `(theta, phi)`를 복원한다.

In [ ]:
"""Synthesize a 4-antenna goniopolarimetric measurement and recover source direction."""


def antenna_unit_vectors(n_antennas: int = 4, tilt_deg: float = 35.0) -> np.ndarray:
    """Build 4 antenna pointings: 3 equispaced in the spin plane + 1 axial.

    Args:
        n_antennas: Number of antennas (must be 4 here).
        tilt_deg: Tilt of axial antenna out of the spin plane in degrees.

    Returns:
        (n_antennas, 3) array of unit vectors in spacecraft frame.
    """
    if n_antennas != 4:
        raise ValueError('This helper is hard-coded for 4 antennas')
    angles = np.deg2rad([0.0, 90.0, 180.0])
    spin_plane = np.column_stack([np.cos(angles), np.sin(angles), np.zeros_like(angles)])
    tilt = np.deg2rad(tilt_deg)
    axial = np.array([np.sin(tilt), 0.0, np.cos(tilt)])
    return np.vstack([spin_plane, axial])


def k_unit_from_angles(theta_rad: float, phi_rad: float) -> np.ndarray:
    """Convert (theta, phi) to a unit propagation vector."""
    return np.array([np.sin(theta_rad) * np.cos(phi_rad),
                     np.sin(theta_rad) * np.sin(phi_rad),
                     np.cos(theta_rad)])


def antenna_response(antennas: np.ndarray, theta_rad: float, phi_rad: float,
                     spin_phase_rad: np.ndarray) -> np.ndarray:
    """Compute scalar antenna response squared (intensity) versus spin phase.

    The response of a short dipole to a wave whose E-field is perpendicular to k
    is proportional to (1 - (e_hat . k_hat)^2). Spinning the spacecraft rotates the
    spin-plane antennas around the body z axis.

    Args:
        antennas: (4, 3) antenna unit vectors at zero spin phase.
        theta_rad, phi_rad: source direction in spacecraft inertial frame.
        spin_phase_rad: Array of spin angles (rad) sampled within one spin period.

    Returns:
        (n_antennas, n_phases) intensity array.
    """
    k_hat = k_unit_from_angles(theta_rad, phi_rad)
    n_phase = spin_phase_rad.size
    response = np.zeros((antennas.shape[0], n_phase))
    for j, phi_s in enumerate(spin_phase_rad):
        cosp, sinp = np.cos(phi_s), np.sin(phi_s)
        rot = np.array([[cosp, -sinp, 0.0], [sinp, cosp, 0.0], [0.0, 0.0, 1.0]])
        ant_rot = antennas @ rot.T
        proj = ant_rot @ k_hat
        response[:, j] = 1.0 - proj**2
    return response


def goniopolarimetry_fit(observed: np.ndarray, antennas: np.ndarray,
                          spin_phase_rad: np.ndarray) -> tuple[float, float, float]:
    """Least-squares fit for source (theta, phi) and amplitude.

    Returns:
        Tuple (theta_rad, phi_rad, amplitude).
    """

    def residuals(params: np.ndarray) -> np.ndarray:
        theta, phi, amp = params
        model = amp * antenna_response(antennas, theta, phi, spin_phase_rad)
        return (model - observed).ravel()

    # Coarse scan to seed the optimizer.
    best = (np.pi / 2, 0.0, observed.mean())
    best_cost = np.inf
    for theta0 in np.deg2rad([20, 60, 90, 120, 160]):
        for phi0 in np.deg2rad([0, 60, 120, 180, 240, 300]):
            cost = np.sum(residuals([theta0, phi0, observed.mean()])**2)
            if cost < best_cost:
                best_cost = cost
                best = (theta0, phi0, observed.mean())
    res = least_squares(residuals, x0=list(best),
                         bounds=([0.0, -np.pi, 0.0], [np.pi, 2 * np.pi, np.inf]))
    return float(res.x[0]), float(res.x[1]), float(res.x[2])


antennas4 = antenna_unit_vectors(4)
spin_phases = np.linspace(0.0, 2 * np.pi, 64, endpoint=False)

true_theta, true_phi = np.deg2rad(70.0), np.deg2rad(110.0)
true_amp = 1.0
clean = true_amp * antenna_response(antennas4, true_theta, true_phi, spin_phases)
noise_level = 0.05
observed = clean + noise_level * rng.standard_normal(clean.shape)

theta_fit, phi_fit, amp_fit = goniopolarimetry_fit(observed, antennas4, spin_phases)
print('True   (theta, phi):', f'({np.rad2deg(true_theta):.1f}°, {np.rad2deg(true_phi):.1f}°)')
print('Fitted (theta, phi):', f'({np.rad2deg(theta_fit):.1f}°, {np.rad2deg(phi_fit):.1f}°)')
print(f'Recovered amplitude: {amp_fit:.3f} (true 1.000)')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for i, label in enumerate(['Ex+', 'Ey+', 'Ex-', 'Ez (axial)']):
    axes[0].plot(np.rad2deg(spin_phases), observed[i], 'o', ms=3, alpha=0.5,
                  label=f'{label} obs')
    axes[0].plot(np.rad2deg(spin_phases),
                  amp_fit * antenna_response(antennas4, theta_fit, phi_fit, spin_phases)[i],
                  '-', lw=1.5, label=f'{label} fit')
axes[0].set_xlabel('Spin phase (deg)')
axes[0].set_ylabel('Antenna response (a.u.)')
axes[0].set_title('Goniopolarimetric spin modulation')
axes[0].legend(fontsize=7, ncol=2, loc='lower right')
axes[0].grid(True, alpha=0.3)

# Sweep noise level to show angular accuracy scaling.
noise_levels = np.array([0.01, 0.025, 0.05, 0.1, 0.2, 0.4])
errors_deg = []
for nl in noise_levels:
    trial_errors = []
    for _ in range(20):
        obs = clean + nl * rng.standard_normal(clean.shape)
        th, ph, _ = goniopolarimetry_fit(obs, antennas4, spin_phases)
        cos_sep = (np.sin(th) * np.sin(true_theta) * np.cos(ph - true_phi)
                   + np.cos(th) * np.cos(true_theta))
        trial_errors.append(np.rad2deg(np.arccos(np.clip(cos_sep, -1.0, 1.0))))
    errors_deg.append(np.median(trial_errors))
axes[1].loglog(noise_levels, errors_deg, 'o-')
axes[1].set_xlabel('Noise level (a.u.)')
axes[1].set_ylabel('Median angular error (deg)')
axes[1].set_title('Angular precision vs SNR')
axes[1].grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## Part 4: IP Shock Arrival-Time Prediction from Type II Onset / II형 폭발 시작으로부터 우주환경 충격파 도착 시각 예측

**English.** A type II radio burst at frequency `f` originates from a CME-driven shock currently at heliocentric distance `r(f)` (where `f_p(r) = f` or `f / 2` for the harmonic). By tracking the type II frequency drift `df/dt`, we infer the shock speed `v_sh` and extrapolate to L1. WAVES enabled this kind of forecasting *in the previously poorly covered 1–14 MHz band*.

Given two `(t_i, f_i)` points along the type II lane, we invert each `f_i` to `r_i` using the density model from Part 1 and fit `r(t) = r_0 + v_sh (t - t_0)` to predict `t_arrival` when `r = AU_RSUN`.

**Korean.** 주파수 `f`의 II형 전파폭발은 현재 일심거리 `r(f)`에 있는 CME 주도 충격파에서 온다(`f_p(r) = f` 또는 고조파 경우 `f/2`). II형의 `df/dt`를 추적하면 충격파 속도 `v_sh`를 알 수 있고, L1까지 외삽하여 도착 시각을 예측한다. 이 관측은 이전에 커버리지가 약했던 1–14 MHz 대역에서 WAVES가 최초로 가능하게 한 일이다.

In [ ]:
"""Predict IP-shock L1 arrival time from a sparse type II frequency drift."""

from scipy.optimize import brentq


def freq_to_radius(f_khz: float, harmonic: bool = True,
                    r_lo: float = 1.5, r_hi: float = AU_RSUN) -> float:
    """Invert plasma-frequency relation: given f, find r such that f_p(r) = f or f/2.

    Args:
        f_khz: Observed type II frequency (kHz).
        harmonic: If True, assume harmonic emission so f_p = f / 2.
        r_lo, r_hi: Search bracket in R_sun.

    Returns:
        Radius in R_sun.
    """
    target_fp = f_khz / 2.0 if harmonic else f_khz

    def func(r: float) -> float:
        return plasma_frequency_khz(coronal_density(r)) - target_fp

    f_lo, f_hi = func(r_lo), func(r_hi)
    if f_lo * f_hi > 0:
        return r_lo if abs(f_lo) < abs(f_hi) else r_hi
    return brentq(func, r_lo, r_hi)


def predict_shock_arrival(times_s: np.ndarray, freqs_khz: np.ndarray,
                           harmonic: bool = True) -> dict:
    """Estimate shock speed and L1 arrival time from a type II drift trace.

    Args:
        times_s: Times of type II measurements (s, since onset).
        freqs_khz: Type II frequencies (kHz) at those times.
        harmonic: Whether to invert via the harmonic (f_p = f/2) or fundamental.

    Returns:
        Dict with shock_speed_kms, arrival_time_s_from_onset, fitted radii.
    """
    radii_rsun = np.array([freq_to_radius(f, harmonic=harmonic) for f in freqs_khz])
    radii_km = radii_rsun * R_SUN_KM
    coeffs = np.polyfit(times_s, radii_km, 1)  # km / s, km
    v_sh_kms, r0_km = coeffs[0], coeffs[1]
    t_arrival = (AU_KM - r0_km) / v_sh_kms
    return {
        'shock_speed_kms': v_sh_kms,
        'arrival_time_s': t_arrival,
        'arrival_time_hours': t_arrival / 3600.0,
        'fitted_radii_rsun': radii_rsun,
        'r0_km': r0_km,
    }


# Build a synthetic type II observation: 1500 km/s shock from r0 = 4 R_sun.
true_v_shock = 1500.0  # km/s
true_r0 = 4.0  # R_sun
obs_times_s = np.array([0.0, 300.0, 900.0, 1800.0, 3600.0])  # 0, 5, 15, 30, 60 min
obs_radii_rsun = true_r0 + true_v_shock * obs_times_s / R_SUN_KM
obs_n_e = coronal_density(obs_radii_rsun)
obs_fp_khz = plasma_frequency_khz(obs_n_e)
obs_f_khz = 2.0 * obs_fp_khz  # harmonic emission
obs_f_khz_noisy = obs_f_khz * (1.0 + 0.03 * rng.standard_normal(obs_f_khz.shape))

result = predict_shock_arrival(obs_times_s, obs_f_khz_noisy, harmonic=True)
print('Type II onset frequencies (kHz, noisy):', np.round(obs_f_khz_noisy, 1))
print(f'Predicted shock speed: {result["shock_speed_kms"]:.0f} km/s (true {true_v_shock:.0f})')
print(f'Predicted L1 arrival : {result["arrival_time_hours"]:.1f} h after type II onset')
true_arrival_h = (AU_KM - true_r0 * R_SUN_KM) / true_v_shock / 3600.0
print(f'True L1 arrival      : {true_arrival_h:.1f} h')
print(f'Forecast error       : {abs(result["arrival_time_hours"] - true_arrival_h) * 60:.1f} min')

# Visualize the inversion.
t_dense_s = np.linspace(0.0, true_arrival_h * 3600.0, 400)
r_true_km = true_r0 * R_SUN_KM + true_v_shock * t_dense_s
r_fit_km = result['r0_km'] + result['shock_speed_kms'] * t_dense_s
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(t_dense_s / 3600.0, r_true_km / R_SUN_KM, 'k-', lw=1.2, label='True trajectory')
ax.plot(t_dense_s / 3600.0, r_fit_km / R_SUN_KM, 'r--', lw=1.2, label='Fit (from type II f drift)')
ax.plot(obs_times_s / 3600.0, result['fitted_radii_rsun'], 'bo', ms=8,
         label='Inverted r(f_obs)')
ax.axhline(AU_RSUN, color='goldenrod', ls=':', label='1 AU = 215 R$_\\odot$')
ax.axvline(true_arrival_h, color='k', ls=':', alpha=0.5)
ax.axvline(result['arrival_time_hours'], color='r', ls=':', alpha=0.5)
ax.set_xlabel('Time since type II onset (h)')
ax.set_ylabel('Heliocentric distance (R$_\\odot$)')
ax.set_title('IP shock L1 arrival prediction from type II frequency drift')
ax.set_yscale('log')
ax.legend(loc='lower right')
ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.show()

## Summary / 요약

**English.** This notebook reproduces in miniature four scientific products of WIND/WAVES described by Bougeret et al. (1995):
1. The type II/III dynamic-spectrum image is the *signature view* of WAVES across RAD2 → RAD1 → TNR.
2. Plasma-line tracking on TNR-like data demonstrates the role of the onboard neural network as a *peak-finding compressor*.
3. Goniopolarimetry from a 4-antenna spinning array recovers source angles to sub-degree precision at SNR > 20.
4. Type II frequency drift inversion through the coronal density model predicts L1 shock arrival to within tens of minutes — the operational space-weather use of WAVES that drove later real-time forecasting.

**Korean.** 이 노트북은 Bougeret 등(1995)이 기술한 WIND/WAVES의 대표 과학 결과 네 가지를 축소 재현한다: II/III형 우주환경 동적 스펙트럼, TNR 플라즈마 라인 추적, 4안테나 goniopolarimetry, 우주환경 충격파 도착 시각 예측.

| Concept / 개념 | This Paper / 이 논문 | Modern Equivalent / 현대 동등물 |
|---|---|---|
| Multi-band radio receiver | TNR + RAD1 + RAD2 + FFT + TDS | PSP/FIELDS RFS, Solar Orbiter/RPW TNR-HFR |
| Floating-point ADC | Cascaded ×16 amps + 12-bit ADC → 144 dB DR | Modern 24-bit delta-sigma ADCs |
| Onboard neural net (TNR) | Integer LUT feed-forward NN — plasma-line picker | Edge-AI peak detectors on PSP/SO |
| Wavelet-like FIR bank | 32 log channels per band | Continuous wavelet transform in post-processing |
| Spin-modulation gonio. | SUM-mode synthesised inclined dipole | STEREO/WAVES stereoscopic radio imaging |
| Type II shock forecast | f_p inversion via Saito-Newkirk model | ENLIL + DBM ensemble forecasts at CCMC |
